# Capítulo 6: Dados: Tipos, Dados Retangulares e pandas

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-elementos-de-dados-estruturados.html) | Elementos de Dados Estruturados |
| [6.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/02-dados-retangulares.html) | Dados Retangulares |
| [6.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/03-lendo-e-tipando-um-arquivo-real.html) | Lendo e Tipando um Arquivo Real |
| [6.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/04-limpando-e-transformando.html) | Limpando e Transformando |
| [6.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/05-agrupando-e-resumindo.html) | Agrupando e Resumindo |
| [6.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/06-da-tabela-para-o-modelo.html) | Da Tabela para o Modelo |

## Elementos de Dados Estruturados

Todo modelo dos capítulos anteriores recebeu números prontos: um vetor de entrada, um alvo, um gradiente que decrescia passo a passo. Dado bruto não chega assim. Chega em uma tabela, com uma coluna que conta pessoas, outra que mede uma taxa, outra que nomeia um lugar — e cada uma pede um tratamento diferente antes de qualquer conta começar. A primeira pergunta, antes de qualquer ajuste, é de que tipo é cada coluna.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")
pd.set_option("display.max_columns", None)

### A taxonomia dos dados

Dado estruturado se divide em duas famílias. **Numérico** é o que se mede: contínuo é qualquer valor dentro de um intervalo — uma taxa, um comprimento, um preço —, discreto é contagem, sempre inteiro — número de filhos, de vendas, de acessos. **Categórico** é o que se rotula: nominal não tem ordem — uma cor, a sigla de um estado, o nome de uma cidade —, ordinal tem ordem mas não tem distância — uma faixa "baixa/média/alta" diz que alta vem depois de média, mas não diz *quanto* maior —, e binário é o caso particular de só dois valores, do tipo "sim ou não".

> **🔷 Conceito**
>
> | Família | Subtipo | Tem ordem? | Tem distância? | Exemplo |
> |---|---|---|---|---|
> | Numérico | Contínuo | sim | sim | uma taxa, um preço |
> | Numérico | Discreto | sim | sim | uma contagem |
> | Categórico | Nominal | não | não | uma sigla, uma cor |
> | Categórico | Ordinal | sim | não | uma faixa "baixa/média/alta" |
> | Categórico | Binário | — | não | "sim"/"não" |
>
> A distância é o que separa numérico de ordinal: em ambos dá para dizer "isto vem depois daquilo", mas só no numérico faz sentido perguntar "quanto depois".

O tipo de uma coluna decide qual gráfico faz sentido para ela, qual estatística pode ser calculada sobre ela e como um programa consegue validar o que entra nela; tratar uma variável ordinal como se fosse numérica produz conclusão errada com aparência de rigor — a média entre "baixa" e "alta" devolve um número, só que um número que não significa nada, porque a distância entre as duas nunca foi definida.

### O que o `pandas` infere sozinho

Para ver a distinção aplicada a um caso concreto, considere as unidades federativas do Brasil, com população e taxa de homicídios:

In [ ]:
estados = pd.read_csv("dados/estados.csv")
estados.dtypes

O `pandas` acerta metade da tarefa sozinho, só olhando a forma dos valores: `Populacao` vira `int64` — é contagem, portanto numérico discreto —, e `Taxa.Homicidios` vira `float64` — é uma taxa, portanto numérico contínuo. `Estado` e `Sigla` viram `object`, o tipo genérico de texto que o `pandas` usa quando não sabe o que mais dizer sobre uma coluna: ele enxerga strings, não que "RO" e "AC" vêm de um conjunto fechado de rótulos possíveis. É inferência sobre a *forma* do valor, não sobre o que ele *significa* — só quem lê o dado sabe que `Sigla` é categórico nominal.

In [ ]:
len(estados), estados["Sigla"].nunique()

As 27 linhas trazem 27 siglas distintas — nenhuma se repete.

### Declarando o categórico

Dizer ao `pandas` o que já se sabe sobre a coluna é uma linha:

In [ ]:
antes = estados.memory_usage(deep=True)
estados["Sigla"] = estados["Sigla"].astype("category")
estados["Sigla"].cat.categories

Antes de acreditar que a conversão economiza alguma coisa, meça. Comparando o consumo de memória coluna a coluna, de antes para depois:

In [ ]:
depois = estados.memory_usage(deep=True)
comparacao = pd.DataFrame({"object": antes, "category": depois})
comparacao.loc["total"] = comparacao.sum()
comparacao["variação (%)"] = (
    (comparacao["category"] - comparacao["object"]) / comparacao["object"] * 100
).round(1)
comparacao

`Sigla` sobe de 1.377 para 2.476 bytes — quase 80% a mais —, e como nenhuma outra coluna muda, o `DataFrame` inteiro sobe de 3.694 para 4.793 bytes, quase 30% a mais. Com 27 valores todos distintos, não há repetição nenhuma para o `category` amortizar: o array de códigos mais o índice de categorias, por cima do que já existia, custam mais do que os ponteiros de texto do `object`. O ganho aqui é **semântico, não de memória** — e, sem repetição para amortizar, nem podia ser: o que muda é que operações que só fazem sentido sobre um conjunto fechado de rótulos passam a existir, como listar as categorias possíveis ou recusar um valor que não está entre elas. A economia de memória aparece quando poucos valores se repetem em muitas linhas — é o caso que a seção 6.3 mostra, ao tipar uma coluna de cidade.

### Ordem sem distância: o categórico ordinal

Nem todo categórico é nominal. Uma faixa de população tem ordem — "6 a 15 milhões" vem depois de "2 a 6 milhões" — mesmo sem ter distância: a faixa não diz o quanto maior. `pd.cut` corta um numérico em faixas categóricas, e `ordered=True` guarda essa ordem em vez de descartá-la:

In [ ]:
faixa = pd.cut(
    estados["Populacao"],
    bins=[0, 2_000_000, 6_000_000, 15_000_000, 50_000_000],
    labels=["até 2 milhões", "2 a 6 milhões", "6 a 15 milhões", "mais de 15 milhões"],
    ordered=True,
)
faixa.value_counts()

Dez estados ficam na faixa intermediária, de 2 a 6 milhões; só três passam de 15 milhões. Com a ordem declarada, comparar deixa de ser comparar texto e passa a ser comparar posição:

In [ ]:
(faixa < "6 a 15 milhões").sum()

Quinze estados ficam abaixo da faixa "6 a 15 milhões" — o `<` olha a posição de cada faixa na ordem que `ordered=True` fixou, não a ordem alfabética dos rótulos (que poria "6 a 15 milhões" antes de "até 2 milhões", e a conta sairia errada).

> **🟩 Exemplo**
>
> Pedir a mesma comparação a uma coluna nominal não tem resposta, e o `pandas` recusa a pergunta em vez de inventar uma:

In [ ]:
try:
    estados["Sigla"] < "SP"
except TypeError as erro:
    print(erro)

> Sem ordem declarada, "menor que" não está definido para `Sigla` — é a mesma pergunta que devolveria um número, e uma falsa sensação de precisão, se a coluna tivesse ficado como texto solto em vez de `category`.

### O tipo escolhe o gráfico

A mesma distinção decide o gráfico. Um categórico com um número por rótulo pede barras — a pergunta é "quanto vale cada um"; um numérico contínuo pede histograma — a pergunta é "como os valores se distribuem":

In [ ]:
# Figura: Um categórico (a unidade federativa) pede barras; um contínuo (a taxa de homicídios) pede histograma — o tipo escolhe o gráfico
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

top10 = estados.nlargest(10, "Populacao").sort_values("Populacao")
ax1.barh(top10["Estado"], top10["Populacao"] / 1e6)
ax1.set_xlabel("população (milhões)")
ax1.set_title("10 maiores populações")

ax2.hist(estados["Taxa.Homicidios"], bins=8)
ax2.set_xlabel("taxa de homicídios (por 100 mil hab.)")
ax2.set_ylabel("estados")
ax2.set_title("distribuição da taxa de homicídios")

plt.tight_layout()
plt.show()

`estados` ainda é só quatro colunas soltas, cada uma com o seu tipo. A próxima seção olha a estrutura que as mantém juntas — o `DataFrame` em si, a sua forma, o seu índice, e as duas formas de escolher linha e coluna dentro dele.

## Dados Retangulares

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Lendo e Tipando um Arquivo Real

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Limpando e Transformando

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Agrupando e Resumindo

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Da Tabela para o Modelo

> **⚠️ Atenção — Em construção**
>
> O conteúdo desta seção ainda será escrito.

## Leituras adicionais

*A escrever.*